# W&B Rerun — Runner 5 of 5

**Assignment:** 5 Medium  
**Experiments:** 5  

1. **vR.P.10** -- CBAM Attention (run01)  
2. **vR.P.10** -- CBAM Attention (reproducibility) (run02)  
3. **vR.P.14** -- Test-Time Augmentation (run01)  
4. **vR.P.14** -- TTA (P.14b complete) (run02)  
5. **vR.P.15** -- Multi-Quality ELA (BEST) (run01)  

Each notebook runs sequentially via **papermill**, logging metrics
to the `Tampered Image Detection & Localization` W&B project in real-time.

In [ ]:
# ============================================================
# Runner 5 of 5 -- Setup
# ============================================================
!pip install -q papermill wandb segmentation-models-pytorch albumentations pyyaml

import os, sys, time, json, yaml
import torch

RUNNER_ID = 5
DATASET_PATH = "/kaggle/input/vrpx-source-notebooks"
CONFIG_PATH = "/kaggle/input/wandb-runner-5-config"
OUTPUT_DIR = "/kaggle/working/executed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Add utilities to path
sys.path.insert(0, os.path.join(CONFIG_PATH, 'utilities'))
from execute_notebook import execute_experiment
from gpu_cleanup import cleanup_gpu
from wandb_utils import setup_wandb

# Authenticate W&B
setup_wandb()

# Verify GPU
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} ({mem:.1f} GB)')
else:
    print('WARNING: No GPU detected!')

# Verify source notebooks exist
if os.path.isdir(DATASET_PATH):
    files = [f for f in os.listdir(DATASET_PATH) if f.endswith('.ipynb')]
    print(f'Source dataset: {len(files)} notebooks found')
else:
    print(f'ERROR: Dataset not found at {DATASET_PATH}')

In [ ]:
# ============================================================
# Runner 5 -- Load Experiment Queue from YAML
# ============================================================
yaml_path = os.path.join(CONFIG_PATH, "runner_5_experiments.yaml")
with open(yaml_path) as f:
    config = yaml.safe_load(f)

EXPERIMENTS = config["experiments"]

print(f'\nRunner {RUNNER_ID}: {len(EXPERIMENTS)} experiments queued')
print('=' * 60)
for i, exp in enumerate(EXPERIMENTS, 1):
    src = os.path.join(DATASET_PATH, exp["file"])
    exists = 'OK' if os.path.exists(src) else 'MISSING'
    print(f'  {i}. [{exists:>7}] {exp["version"]:12} {exp["desc"]} ({exp["run_id"]})')

In [ ]:
# ============================================================
# Runner 5 -- Sequential Execution
# ============================================================
results = []
run_start = time.time()

for idx, exp in enumerate(EXPERIMENTS, 1):
    nb_path = os.path.join(DATASET_PATH, exp["file"])
    out_name = f"{exp['version'].replace('.', '')}_run01_output.ipynb"
    out_path = os.path.join(OUTPUT_DIR, out_name)

    print(f'\n{"=" * 70}')
    print(f'  [{idx}/{len(EXPERIMENTS)}] {exp["version"]} -- {exp["desc"]} ({exp["run_id"]})')
    print(f'{"=" * 70}')

    status, elapsed, error = execute_experiment(nb_path, out_path)

    result = {**exp, 'status': status, 'time_min': elapsed}
    if error:
        result['error'] = error
        print(f'  {status} after {elapsed:.1f} min: {error[:100]}')
    else:
        print(f'  {status} in {elapsed:.1f} min')

    results.append(result)

    # GPU cleanup between experiments
    cleanup_gpu()
    print(f'  GPU cache cleared')

total_min = (time.time() - run_start) / 60
print(f'\nAll experiments processed. Total wall time: {total_min:.1f} min ({total_min/60:.1f} h)')

In [ ]:
# ============================================================
# Runner 5 -- Execution Summary
# ============================================================
print(f'\n{"=" * 70}')
print(f'  RUNNER 5 COMPLETE')
print(f'{"=" * 70}\n')

gpu_total = sum(r['time_min'] for r in results)
ok = [r for r in results if r['status'] == 'SUCCESS']
fail = [r for r in results if r['status'] != 'SUCCESS']

for r in results:
    icon = 'OK' if r['status'] == 'SUCCESS' else r['status']
    print(f"  [{icon:>9}] {r['version']:12} {r['desc']:40} {r['time_min']:6.1f} min")
    if 'error' in r:
        print(f"             {r['error'][:100]}")

print(f'\n  Succeeded: {len(ok)}/{len(results)}')
if fail:
    print(f'  Failed:    {len(fail)}/{len(results)}')
print(f'  GPU time:  {gpu_total:.1f} min ({gpu_total/60:.1f} h)')

# Save JSON summary
summary_path = os.path.join(OUTPUT_DIR, f'runner_5_summary.json')
with open(summary_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'\n  Summary saved: {summary_path}')

# List output notebooks
print(f'\n  Output notebooks:')
for f_name in sorted(os.listdir(OUTPUT_DIR)):
    if f_name.endswith('.ipynb'):
        size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f_name)) / 1e6
        print(f'    {f_name} ({size_mb:.1f} MB)')